# 01 · Tensors, Autograd & Core Mechanics

| | |
|---|---|
| **Author** | Chandan C |
| **GitHub** | [github.com/chandanc5525](https://github.com/chandanc5525) |
| **Course** | Agentic AI BootCamp · Module 02 – Advanced PyTorch |
| **Repository** | [AgenticAI_BootCamp-/02_advanced_pytorch](https://github.com/chandanc5525/AgenticAI_BootCamp-/tree/main/02_advanced_pytorch) |
| **Level** | Beginner |
| **Prerequisites** | Basic Python and NumPy |

---

### What you will learn
- Creating tensors, dtypes, devices
- Indexing, slicing, reshaping, broadcasting
- Reductions, matmul, einsum
- NumPy interop and memory sharing
- Autograd, computation graphs, custom `autograd.Function`
- Gradient descent written from scratch

> Every cell runs on a plain CPU. If a GPU (CUDA / Apple MPS) is present, the notebooks use it automatically.

## 0. Setup

In [ ]:
import math, random
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
    else "cpu"
)
print("PyTorch:", torch.__version__, "| device:", device)

## 1. Creating tensors
A **tensor** is an n-dimensional array with a dtype, a device and (optionally) gradient tracking. Everything in PyTorch – images, token ids, embeddings, model weights – is a tensor.

In [ ]:
a = torch.tensor([[1, 2, 3], [4, 5, 6]])           # from Python data
b = torch.zeros(2, 3)                               # zeros
c = torch.ones(2, 3)                                # ones
d = torch.full((2, 3), 7.0)                         # constant
e = torch.arange(0, 10, 2)                          # range
f = torch.linspace(0, 1, 5)                         # evenly spaced
g = torch.eye(3)                                    # identity
r = torch.rand(2, 3)                                # uniform [0, 1)
n = torch.randn(2, 3)                               # standard normal
i = torch.randint(0, 10, (2, 3))                    # random ints
like = torch.zeros_like(a, dtype=torch.float32)     # same shape as another tensor

for name, t in dict(a=a, b=b, e=e, f=f, g=g, i=i, like=like).items():
    print(name, tuple(t.shape), t.dtype)

## 2. Attributes: shape, dtype, device
Mismatched **dtype** or **device** are the two most common runtime errors you will meet.

In [ ]:
x = torch.randn(3, 4)
print("shape :", x.shape, "| ndim:", x.ndim, "| numel:", x.numel())
print("dtype :", x.dtype)
print("device:", x.device)

# Casting and moving
x_half = x.to(torch.float16)
x_int  = x.long()
x_dev  = x.to(device)
print(x_half.dtype, x_int.dtype, x_dev.device)

# Common dtypes in GenAI work
#  float32 -> default training | bfloat16/float16 -> mixed precision & LLM inference
#  int64   -> token ids        | bool -> attention masks
print(torch.tensor([1, 2, 3]).dtype, torch.tensor([1., 2.]).dtype, torch.tensor([True]).dtype)

## 3. Indexing and slicing

In [ ]:
x = torch.arange(24).reshape(4, 6)
print(x)
print("row 0        :", x[0])
print("column 1     :", x[:, 1])
print("sub-matrix   :", x[1:3, 2:5].tolist())
print("last row     :", x[-1])
print("every 2nd col:", x[:, ::2].shape)

# Boolean masking (used for attention masks / padding)
mask = x % 5 == 0
print("mask select  :", x[mask])
x2 = x.clone(); x2[x2 > 20] = -1
print("after masked assignment:\n", x2)

# Fancy indexing & gather (selecting log-probs of target tokens!)
idx = torch.tensor([0, 2, 3])
print("rows [0,2,3] :", x[idx].shape)
logits = torch.randn(3, 5)
targets = torch.tensor([1, 4, 0])
print("gather       :", logits.gather(1, targets.unsqueeze(1)).squeeze(1))

## 4. Reshaping: `view`, `reshape`, `permute`, `squeeze`, `cat`, `stack`
- `view` needs contiguous memory and shares storage. `reshape` copies only if needed.
- `permute` / `transpose` reorder dims (non-contiguous result).
- Transformers constantly do `(B, T, C) -> (B, heads, T, C/heads)`.

In [ ]:
x = torch.arange(24)
print(x.view(2, 3, 4).shape, x.reshape(4, -1).shape)

B, T, C, H = 2, 5, 8, 4
t = torch.randn(B, T, C)
heads = t.view(B, T, H, C // H).transpose(1, 2)    # (B, H, T, C/H)
print("multi-head layout:", heads.shape, "| contiguous:", heads.is_contiguous())
merged = heads.transpose(1, 2).contiguous().view(B, T, C)
print("merged back      :", merged.shape, "| equal:", torch.equal(t, merged))

v = torch.zeros(1, 3, 1)
print(v.squeeze().shape, v.squeeze(0).shape, torch.zeros(3).unsqueeze(0).shape)

p, q = torch.ones(2, 3), torch.zeros(2, 3)
print("cat  :", torch.cat([p, q], dim=0).shape)     # joins existing dim
print("stack:", torch.stack([p, q], dim=0).shape)   # creates new dim

## 5. Math, broadcasting and reductions
**Broadcasting** aligns shapes from the right; a dimension of size 1 is stretched.

In [ ]:
a = torch.tensor([[1.], [2.], [3.]])      # (3,1)
b = torch.tensor([10., 20., 30., 40.])    # (4,)
print("broadcast sum:", (a + b).shape)    # -> (3,4)

x = torch.randn(4, 5)
print("sum/mean/std:", x.sum().item(), x.mean(0).shape, x.std(1).shape)
print("max (values, idx):", [t.shape for t in x.max(dim=1)])
print("argmax:", x.argmax(dim=1), "| topk:", x.topk(2, dim=1).indices.shape)

# Softmax & numerical stability
logits = torch.tensor([1000., 1001., 1002.])
naive = torch.exp(logits) / torch.exp(logits).sum()
stable = torch.softmax(logits, dim=0)
print("naive :", naive, "\nstable:", stable)

# L2 normalisation (needed for cosine similarity in RAG)
emb = torch.randn(3, 8)
emb_n = torch.nn.functional.normalize(emb, dim=-1)
print("norms:", emb_n.norm(dim=-1))

## 6. Matrix multiplication and `einsum`

In [ ]:
A, Bm = torch.randn(3, 4), torch.randn(4, 5)
print((A @ Bm).shape, torch.allclose(A @ Bm, torch.matmul(A, Bm)))

# batched matmul – the heart of attention
q = torch.randn(2, 4, 6, 8)      # (B, H, T, d)
k = torch.randn(2, 4, 6, 8)
scores = q @ k.transpose(-2, -1) # (B, H, T, T)
print("attention scores:", scores.shape)

# einsum: explicit and readable
scores2 = torch.einsum("bhid,bhjd->bhij", q, k)
print("einsum matches matmul:", torch.allclose(scores, scores2, atol=1e-6))
print("outer product:", torch.einsum("i,j->ij", torch.arange(3.), torch.arange(4.)).shape)

## 7. NumPy interoperability
`torch.from_numpy` and `.numpy()` **share memory** on CPU - modifying one changes the other.

In [ ]:
arr = np.array([1., 2., 3.])
t = torch.from_numpy(arr)
t[0] = 100
print("numpy changed too:", arr)

t2 = torch.tensor(arr)       # copies
t2[0] = -1
print("no change:", arr)

print(type(torch.ones(2).numpy()))
# a tensor with grad must be detached first
w = torch.ones(2, requires_grad=True)
print(w.detach().numpy())

## 8. Autograd – automatic differentiation
PyTorch records operations on tensors with `requires_grad=True` in a **dynamic computation graph**; `.backward()` walks the graph in reverse (chain rule) and stores gradients in `.grad`.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
w = torch.tensor(2.0, requires_grad=True)
y = w * x**2 + 5          # y = w x^2 + 5
y.backward()
print("dy/dx = 2wx =", x.grad.item(), "| dy/dw = x^2 =", w.grad.item())
print("graph node:", y.grad_fn)

# Gradients ACCUMULATE – always zero them
x.grad.zero_()
(x**3).backward()
print("d/dx x^3 =", x.grad.item())

# Turning tracking off (inference, evaluation, target networks)
with torch.no_grad():
    z = x * 2
print("requires_grad inside no_grad:", z.requires_grad)
print("detach:", (x * 2).detach().requires_grad)
with torch.inference_mode():            # even leaner than no_grad
    print("inference_mode:", (x * 2).requires_grad)

### 8.1 Vector outputs, `torch.autograd.grad` and higher-order gradients

In [ ]:
x = torch.linspace(-2, 2, 5, requires_grad=True)
y = (x ** 3).sum()                                  # backward() needs a scalar
(g,) = torch.autograd.grad(y, x, create_graph=True) # g = 3x^2, keep graph
(g2,) = torch.autograd.grad(g.sum(), x)             # d/dx 3x^2 = 6x
print("x  :", x.detach())
print("g  :", g.detach())
print("g2 :", g2)

# Jacobian with torch.func
from torch.func import jacrev
f = lambda v: torch.stack([v[0]**2, v[0]*v[1]])
print("Jacobian:\n", jacrev(f)(torch.tensor([2., 3.])))

### 8.2 Custom `autograd.Function`
Useful for non-differentiable ops, straight-through estimators (quantisation), or custom CUDA kernels.

In [ ]:
class ClampedReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, inp, max_val):
        ctx.save_for_backward(inp)
        ctx.max_val = max_val
        return inp.clamp(min=0, max=max_val)

    @staticmethod
    def backward(ctx, grad_out):
        (inp,) = ctx.saved_tensors
        grad = grad_out.clone()
        grad[(inp < 0) | (inp > ctx.max_val)] = 0
        return grad, None

x = torch.tensor([-1., 0.5, 2., 7.], requires_grad=True)
out = ClampedReLU.apply(x, 6.0)
out.sum().backward()
print(out, x.grad)
print("gradcheck:", torch.autograd.gradcheck(lambda t: ClampedReLU.apply(t, 6.0),
      torch.tensor([0.5, 2.0], dtype=torch.double, requires_grad=True)))

## 9. Putting it together – linear regression with manual gradient descent
Before `nn.Module` and optimisers, see what they automate.

In [ ]:
torch.manual_seed(0)
X = torch.randn(200, 1)
y = 3.0 * X + 0.5 + 0.1 * torch.randn(200, 1)       # true w=3, b=0.5

w = torch.zeros(1, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.1

for epoch in range(100):
    pred = X @ w + b
    loss = ((pred - y) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_(); b.grad.zero_()
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():.4f}")
print(f"learned w={w.item():.3f}, b={b.item():.3f}")

## 10. Memory & performance habits
- Prefer in-place ops only when you are sure autograd doesn't need the old value.
- `tensor.item()` / `.cpu()` force GPU sync – avoid them in hot loops.
- Use `torch.no_grad()` / `inference_mode()` for evaluation and generation.

In [ ]:
import time
x = torch.randn(1024, 1024)
t0 = time.perf_counter()
for _ in range(10): x @ x
print(f"10 matmuls on CPU: {time.perf_counter()-t0:.3f}s")
print("threads:", torch.get_num_threads())
print("element size (bytes):", x.element_size(), "| memory MB:", x.numel()*x.element_size()/1e6)

## Exercises
1. Build a `(5, 3)` tensor and extract all rows whose sum is greater than the mean row-sum.
2. Implement `softmax` from scratch with the max-subtraction trick and compare with `torch.softmax`.
3. Compute cosine similarity between a query vector and 10 document vectors using only tensor ops.
4. Extend the manual regression to *polynomial* features (`x, x^2`) and recover the coefficients.
5. Write a custom `autograd.Function` for a straight-through sign function.

---
### Summary & next step
Work through the exercises above before moving on. Next notebook: **02_nn_module_training_pipeline.ipynb**.

*Notebook by **Chandan C** · [github.com/chandanc5525](https://github.com/chandanc5525) · Agentic AI BootCamp*